# NB06 – Gold to Warehouse

## Objective

This notebook publishes the curated Gold Layer data into the Microsoft Fabric Warehouse (`WH_Retail`).

The Warehouse serves as the enterprise reporting database and is optimized for SQL analytics and Power BI reporting.

### Activities Performed

- Read Gold Layer tables
- Load Dimension tables into the `dim` schema
- Load Fact tables into the `fact` schema
- Load Aggregated Reporting tables into the `rpt` schema
- Validate the Warehouse load
- Execute SQL analytical queries

### Source

- LH_Gold

### Destination

- WH_Retail

## Import Required Libraries

Import the PySpark libraries required for reading Delta tables and preparing data before loading it into the Fabric Warehouse.

In [3]:
# ============================================================
# Import Required Libraries
# ============================================================

from pyspark.sql import functions as F

print("Libraries imported successfully.")

StatementMeta(, ec4e6b8d-15b1-4ec4-9d56-f785f0797b2f, 5, Finished, Available, Finished, False)

Libraries imported successfully.


## Configure Environment

Define the Lakehouse location and Warehouse details used throughout the notebook.

In [4]:
# ============================================================
# Configuration
# ============================================================

gold_base_path = (
    "abfss://EnterpriseRetailAnalytics@onelake.dfs.fabric.microsoft.com/"
    "LH_Gold.Lakehouse/Tables/dbo"
)

warehouse_name = "WH_Retail"

print("Configuration completed.")

StatementMeta(, ec4e6b8d-15b1-4ec4-9d56-f785f0797b2f, 6, Finished, Available, Finished, False)

Configuration completed.


## Read Gold Layer Tables

Read all curated Gold tables from the Gold Lakehouse into Spark DataFrames.

These DataFrames will be loaded into the Fabric Warehouse.

In [5]:
# ============================================================
# Read Dimension Tables
# ============================================================

dim_customer = spark.read.format("delta").load(f"{gold_base_path}/dimcustomer")
dim_product = spark.read.format("delta").load(f"{gold_base_path}/dimproduct")
dim_store = spark.read.format("delta").load(f"{gold_base_path}/dimstore")
dim_region = spark.read.format("delta").load(f"{gold_base_path}/dimregion")
dim_supplier = spark.read.format("delta").load(f"{gold_base_path}/dimsupplier")
dim_employee = spark.read.format("delta").load(f"{gold_base_path}/dimemployee")
dim_promotion = spark.read.format("delta").load(f"{gold_base_path}/dimpromotion")
dim_date = spark.read.format("delta").load(f"{gold_base_path}/dimdate")

# ============================================================
# Read Fact Tables
# ============================================================

fact_sales = spark.read.format("delta").load(f"{gold_base_path}/factsales")
fact_inventory = spark.read.format("delta").load(f"{gold_base_path}/factinventory")
fact_returns = spark.read.format("delta").load(f"{gold_base_path}/factreturns")

# ============================================================
# Read Reporting Tables
# ============================================================

gold_sales_daily = spark.read.format("delta").load(f"{gold_base_path}/goldsalesdaily")
gold_sales_monthly = spark.read.format("delta").load(f"{gold_base_path}/goldsalesmonthly")
gold_product_performance = spark.read.format("delta").load(f"{gold_base_path}/goldproductperformance")
gold_store_performance = spark.read.format("delta").load(f"{gold_base_path}/goldstoreperformance")
gold_customer_performance = spark.read.format("delta").load(f"{gold_base_path}/goldcustomerperformance")
gold_inventory_summary = spark.read.format("delta").load(f"{gold_base_path}/goldinventorysummary")
gold_returns_summary = spark.read.format("delta").load(f"{gold_base_path}/goldreturnssummary")

print("All Gold tables loaded successfully.")

StatementMeta(, ec4e6b8d-15b1-4ec4-9d56-f785f0797b2f, 7, Finished, Available, Finished, False)

All Gold tables loaded successfully.


## Publish Gold Layer to Fabric Warehouse

Load all certified Gold Layer tables into the Fabric Warehouse (`WH_Retail`).

The tables are organized as:

- `dim` → Dimension tables
- `fact` → Fact tables
- `rpt` → Reporting and aggregated tables

The Fabric Spark Data Warehouse Connector performs a scalable bulk load from Spark DataFrames into Warehouse tables.

In [6]:
# ============================================================
# Import Fabric Warehouse Connector
# ============================================================

import com.microsoft.spark.fabric
from com.microsoft.spark.fabric.Constants import Constants

print("Fabric Warehouse Connector Loaded Successfully.")

StatementMeta(, ec4e6b8d-15b1-4ec4-9d56-f785f0797b2f, 8, Finished, Available, Finished, False)

Fabric Warehouse Connector Loaded Successfully.


# Running the below code will throw error as The Fabric Warehouse Connector does not support Spark's `timestamp_ntz` data type.

Before loading data into the Warehouse, convert these columns to the SQL-compatible `DATE` type.

In [1]:
# ============================================================
# Publish Dimension Tables
# ============================================================

dim_customer.write.mode("overwrite").synapsesql("WH_Retail.dim.DimCustomer")

dim_product.write.mode("overwrite").synapsesql("WH_Retail.dim.DimProduct")

dim_store.write.mode("overwrite").synapsesql("WH_Retail.dim.DimStore")

dim_region.write.mode("overwrite").synapsesql("WH_Retail.dim.DimRegion")

dim_supplier.write.mode("overwrite").synapsesql("WH_Retail.dim.DimSupplier")

dim_employee.write.mode("overwrite").synapsesql("WH_Retail.dim.DimEmployee")

dim_promotion.write.mode("overwrite").synapsesql("WH_Retail.dim.DimPromotion")

dim_date.write.mode("overwrite").synapsesql("WH_Retail.dim.DimDate")

print("All Dimension Tables Published Successfully.")

StatementMeta(, cbe5495e-0a65-45f1-8b89-c66ef3bc0697, 3, Finished, Available, Finished, False)

NameError: name 'dim_customer' is not defined

## CHECKING SCHEMA

In [6]:
    dim_date.printSchema()

StatementMeta(, 0c3d14c8-f42d-4f74-8e35-8e2d284313ba, 8, Finished, Available, Finished, False)

root
 |-- DateKey: long (nullable = true)
 |-- Date: timestamp_ntz (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Quarter: string (nullable = true)
 |-- MonthNumber: integer (nullable = true)
 |-- MonthName: string (nullable = true)
 |-- YearMonth: string (nullable = true)
 |-- YearMonthNumber: integer (nullable = true)
 |-- WeekNumber: long (nullable = true)
 |-- Day: integer (nullable = true)
 |-- DayName: string (nullable = true)
 |-- IsWeekend: boolean (nullable = true)
 |-- FiscalYear: integer (nullable = true)



In [7]:
# Check schema of every dataframe

tables = {
    "DimCustomer": dim_customer,
    "DimProduct": dim_product,
    "DimStore": dim_store,
    "DimRegion": dim_region,
    "DimSupplier": dim_supplier,
    "DimEmployee": dim_employee,
    "DimPromotion": dim_promotion,
    "DimDate": dim_date,
    "FactSales": fact_sales,
    "FactInventory": fact_inventory,
    "FactReturns": fact_returns,
    "GoldSalesDaily": gold_sales_daily,
    "GoldSalesMonthly": gold_sales_monthly,
    "GoldProductPerformance": gold_product_performance,
    "GoldStorePerformance": gold_store_performance,
    "GoldCustomerPerformance": gold_customer_performance,
    "GoldInventorySummary": gold_inventory_summary,
    "GoldReturnsSummary": gold_returns_summary,
}

for name, df in tables.items():
    print(f"\n{name}")
    df.printSchema()

StatementMeta(, 0c3d14c8-f42d-4f74-8e35-8e2d284313ba, 9, Finished, Available, Finished, False)


DimCustomer
root
 |-- CustomerKey: long (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- DateOfBirth: date (nullable = true)
 |-- Age: long (nullable = true)
 |-- AgeGroup: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- RegionKey: long (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- JoinDate: date (nullable = true)
 |-- LoyaltyTier: string (nullable = true)


DimProduct
root
 |-- ProductKey: long (nullable = true)
 |-- ProductCode: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- SupplierKey: long (nullable = true)
 |-- UnitCost: double (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- IsActive: boolean (nullable = true)


DimStor

## Convert Unsupported Timestamp Columns

The Fabric Warehouse Connector does not support Spark's `timestamp_ntz` data type.

Before loading data into the Warehouse, convert these columns to the SQL-compatible `DATE` type.

In [7]:
from pyspark.sql.functions import col

# Convert TimestampNTZ → Date

dim_date = dim_date.withColumn(
    "Date",
    col("Date").cast("date")
)

gold_sales_daily = gold_sales_daily.withColumn(
    "Date",
    col("Date").cast("date")
)

print("TimestampNTZ columns converted successfully.")

StatementMeta(, ec4e6b8d-15b1-4ec4-9d56-f785f0797b2f, 9, Finished, Available, Finished, False)

TimestampNTZ columns converted successfully.


## Verify the conversion

In [8]:
dim_date.printSchema()
gold_sales_daily.printSchema()

StatementMeta(, ec4e6b8d-15b1-4ec4-9d56-f785f0797b2f, 10, Finished, Available, Finished, False)

root
 |-- DateKey: long (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Quarter: string (nullable = true)
 |-- MonthNumber: integer (nullable = true)
 |-- MonthName: string (nullable = true)
 |-- YearMonth: string (nullable = true)
 |-- YearMonthNumber: integer (nullable = true)
 |-- WeekNumber: long (nullable = true)
 |-- Day: integer (nullable = true)
 |-- DayName: string (nullable = true)
 |-- IsWeekend: boolean (nullable = true)
 |-- FiscalYear: integer (nullable = true)

root
 |-- SalesDateKey: long (nullable = true)
 |-- TotalSales: double (nullable = true)
 |-- TotalNetSales: double (nullable = true)
 |-- TotalDiscount: double (nullable = true)
 |-- TotalQuantity: long (nullable = true)
 |-- TotalOrders: long (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Quarter: string (nullable = true)
 |-- MonthNumber: integer (nullable = true)
 |-- MonthName: string (nullable = true)
 |-- YearMonth:

## Create Reusable Warehouse Load Function

Instead of writing separate load statements for every table, create a reusable function that publishes any Spark DataFrame to the Fabric Warehouse.

### Benefits

- Reduces duplicate code
- Easier maintenance
- Standardized logging
- Easier debugging
- Reusable for future projects

In [9]:
# ============================================================
# Reusable Warehouse Load Function
# ============================================================

def load_to_warehouse(df, schema_name, table_name, mode="overwrite"):
    """
    Load a Spark DataFrame into the Fabric Warehouse.

    Parameters
    ----------
    df : Spark DataFrame
        DataFrame to be loaded.

    schema_name : str
        Warehouse schema (dim, fact, rpt)

    table_name : str
        Destination table name.

    mode : str
        overwrite | append | ignore | errorifexists
    """

    full_table_name = f"WH_Retail.{schema_name}.{table_name}"

    print(f"Loading {full_table_name} ...")

    (
        df.write
          .mode(mode)
          .synapsesql(full_table_name)
    )

    print(f"✓ Successfully loaded {full_table_name}")

StatementMeta(, ec4e6b8d-15b1-4ec4-9d56-f785f0797b2f, 11, Finished, Available, Finished, False)

## Publish Dimension Tables

Load all certified Dimension tables from the Gold Lakehouse into the `dim` schema of the Fabric Warehouse.

Dimension tables describe business entities and are loaded before Fact tables.

In [10]:
# ============================================================
# Load Dimension Tables
# ============================================================

load_to_warehouse(dim_customer,  "dim", "DimCustomer")
load_to_warehouse(dim_product,   "dim", "DimProduct")
load_to_warehouse(dim_store,     "dim", "DimStore")
load_to_warehouse(dim_region,    "dim", "DimRegion")
load_to_warehouse(dim_supplier,  "dim", "DimSupplier")
load_to_warehouse(dim_employee,  "dim", "DimEmployee")
load_to_warehouse(dim_promotion, "dim", "DimPromotion")
load_to_warehouse(dim_date,      "dim", "DimDate")

print("\nAll Dimension tables loaded successfully.")

StatementMeta(, ec4e6b8d-15b1-4ec4-9d56-f785f0797b2f, 12, Finished, Available, Finished, False)

Loading WH_Retail.dim.DimCustomer ...
✓ Successfully loaded WH_Retail.dim.DimCustomer
Loading WH_Retail.dim.DimProduct ...
✓ Successfully loaded WH_Retail.dim.DimProduct
Loading WH_Retail.dim.DimStore ...
✓ Successfully loaded WH_Retail.dim.DimStore
Loading WH_Retail.dim.DimRegion ...
✓ Successfully loaded WH_Retail.dim.DimRegion
Loading WH_Retail.dim.DimSupplier ...
✓ Successfully loaded WH_Retail.dim.DimSupplier
Loading WH_Retail.dim.DimEmployee ...
✓ Successfully loaded WH_Retail.dim.DimEmployee
Loading WH_Retail.dim.DimPromotion ...
✓ Successfully loaded WH_Retail.dim.DimPromotion
Loading WH_Retail.dim.DimDate ...
✓ Successfully loaded WH_Retail.dim.DimDate

All Dimension tables loaded successfully.


## Publish Fact Tables

Load all curated transactional Fact tables into the `fact` schema.

Fact tables store measurable business events such as sales, inventory, and returns.

In [11]:
# ============================================================
# Load Fact Tables
# ============================================================

load_to_warehouse(fact_sales,      "fact", "FactSales")
load_to_warehouse(fact_inventory,  "fact", "FactInventory")
load_to_warehouse(fact_returns,    "fact", "FactReturns")

print("\nAll Fact tables loaded successfully.")

StatementMeta(, ec4e6b8d-15b1-4ec4-9d56-f785f0797b2f, 13, Finished, Available, Finished, False)

Loading WH_Retail.fact.FactSales ...
✓ Successfully loaded WH_Retail.fact.FactSales
Loading WH_Retail.fact.FactInventory ...
✓ Successfully loaded WH_Retail.fact.FactInventory
Loading WH_Retail.fact.FactReturns ...
✓ Successfully loaded WH_Retail.fact.FactReturns

All Fact tables loaded successfully.


## Publish Reporting Tables

Load the aggregated Gold Layer reporting tables into the `rpt` schema.

These tables are optimized for reporting and dashboard development.

In [12]:
# ============================================================
# Load Reporting Tables
# ============================================================

load_to_warehouse(gold_sales_daily,           "rpt", "GoldSalesDaily")
load_to_warehouse(gold_sales_monthly,         "rpt", "GoldSalesMonthly")
load_to_warehouse(gold_product_performance,   "rpt", "GoldProductPerformance")
load_to_warehouse(gold_store_performance,     "rpt", "GoldStorePerformance")
load_to_warehouse(gold_customer_performance,  "rpt", "GoldCustomerPerformance")
load_to_warehouse(gold_inventory_summary,     "rpt", "GoldInventorySummary")
load_to_warehouse(gold_returns_summary,       "rpt", "GoldReturnsSummary")

print("\nAll Reporting tables loaded successfully.")

StatementMeta(, ec4e6b8d-15b1-4ec4-9d56-f785f0797b2f, 14, Finished, Available, Finished, False)

Loading WH_Retail.rpt.GoldSalesDaily ...
✓ Successfully loaded WH_Retail.rpt.GoldSalesDaily
Loading WH_Retail.rpt.GoldSalesMonthly ...
✓ Successfully loaded WH_Retail.rpt.GoldSalesMonthly
Loading WH_Retail.rpt.GoldProductPerformance ...
✓ Successfully loaded WH_Retail.rpt.GoldProductPerformance
Loading WH_Retail.rpt.GoldStorePerformance ...
✓ Successfully loaded WH_Retail.rpt.GoldStorePerformance
Loading WH_Retail.rpt.GoldCustomerPerformance ...
✓ Successfully loaded WH_Retail.rpt.GoldCustomerPerformance
Loading WH_Retail.rpt.GoldInventorySummary ...
✓ Successfully loaded WH_Retail.rpt.GoldInventorySummary
Loading WH_Retail.rpt.GoldReturnsSummary ...
✓ Successfully loaded WH_Retail.rpt.GoldReturnsSummary

All Reporting tables loaded successfully.


## Validate Warehouse Load

Validate that all Gold Layer tables have been successfully published into the Fabric Warehouse.

Validation includes:

- Row count comparison
- Table existence verification
- Load summary

In [13]:
# ============================================================
# Read Warehouse Tables
# ============================================================

wh_dim_customer = spark.read.synapsesql("WH_Retail.dim.DimCustomer")
wh_dim_product = spark.read.synapsesql("WH_Retail.dim.DimProduct")
wh_dim_store = spark.read.synapsesql("WH_Retail.dim.DimStore")
wh_dim_region = spark.read.synapsesql("WH_Retail.dim.DimRegion")
wh_dim_supplier = spark.read.synapsesql("WH_Retail.dim.DimSupplier")
wh_dim_employee = spark.read.synapsesql("WH_Retail.dim.DimEmployee")
wh_dim_promotion = spark.read.synapsesql("WH_Retail.dim.DimPromotion")
wh_dim_date = spark.read.synapsesql("WH_Retail.dim.DimDate")

wh_fact_sales = spark.read.synapsesql("WH_Retail.fact.FactSales")
wh_fact_inventory = spark.read.synapsesql("WH_Retail.fact.FactInventory")
wh_fact_returns = spark.read.synapsesql("WH_Retail.fact.FactReturns")

wh_gold_sales_daily = spark.read.synapsesql("WH_Retail.rpt.GoldSalesDaily")
wh_gold_sales_monthly = spark.read.synapsesql("WH_Retail.rpt.GoldSalesMonthly")
wh_gold_product_performance = spark.read.synapsesql("WH_Retail.rpt.GoldProductPerformance")
wh_gold_store_performance = spark.read.synapsesql("WH_Retail.rpt.GoldStorePerformance")
wh_gold_customer_performance = spark.read.synapsesql("WH_Retail.rpt.GoldCustomerPerformance")
wh_gold_inventory_summary = spark.read.synapsesql("WH_Retail.rpt.GoldInventorySummary")
wh_gold_returns_summary = spark.read.synapsesql("WH_Retail.rpt.GoldReturnsSummary")

print("Warehouse tables loaded successfully.")

StatementMeta(, ec4e6b8d-15b1-4ec4-9d56-f785f0797b2f, 15, Finished, Available, Finished, False)

Warehouse tables loaded successfully.


In [14]:
# ============================================================
# Row Count Validation
# ============================================================

def validate_row_count(table_name, gold_df, warehouse_df):

    gold_count = gold_df.count()
    warehouse_count = warehouse_df.count()

    status = "PASS" if gold_count == warehouse_count else "FAIL"

    print(
        f"{table_name:<30}"
        f" Gold: {gold_count:<10}"
        f" Warehouse: {warehouse_count:<10}"
        f" {status}"
    )

StatementMeta(, ec4e6b8d-15b1-4ec4-9d56-f785f0797b2f, 16, Finished, Available, Finished, False)

In [15]:
print("=" * 90)
print("WAREHOUSE ROW COUNT VALIDATION")
print("=" * 90)

validate_row_count("DimCustomer", dim_customer, wh_dim_customer)
validate_row_count("DimProduct", dim_product, wh_dim_product)
validate_row_count("DimStore", dim_store, wh_dim_store)
validate_row_count("DimRegion", dim_region, wh_dim_region)
validate_row_count("DimSupplier", dim_supplier, wh_dim_supplier)
validate_row_count("DimEmployee", dim_employee, wh_dim_employee)
validate_row_count("DimPromotion", dim_promotion, wh_dim_promotion)
validate_row_count("DimDate", dim_date, wh_dim_date)

validate_row_count("FactSales", fact_sales, wh_fact_sales)
validate_row_count("FactInventory", fact_inventory, wh_fact_inventory)
validate_row_count("FactReturns", fact_returns, wh_fact_returns)

validate_row_count("GoldSalesDaily", gold_sales_daily, wh_gold_sales_daily)
validate_row_count("GoldSalesMonthly", gold_sales_monthly, wh_gold_sales_monthly)
validate_row_count("GoldProductPerformance", gold_product_performance, wh_gold_product_performance)
validate_row_count("GoldStorePerformance", gold_store_performance, wh_gold_store_performance)
validate_row_count("GoldCustomerPerformance", gold_customer_performance, wh_gold_customer_performance)
validate_row_count("GoldInventorySummary", gold_inventory_summary, wh_gold_inventory_summary)
validate_row_count("GoldReturnsSummary", gold_returns_summary, wh_gold_returns_summary)

print("=" * 90)
print("VALIDATION COMPLETED")
print("=" * 90)

StatementMeta(, ec4e6b8d-15b1-4ec4-9d56-f785f0797b2f, 17, Finished, Available, Finished, False)

WAREHOUSE ROW COUNT VALIDATION
DimCustomer                    Gold: 50000      Warehouse: 50000      PASS
DimProduct                     Gold: 5000       Warehouse: 5000       PASS
DimStore                       Gold: 100        Warehouse: 100        PASS
DimRegion                      Gold: 6          Warehouse: 6          PASS
DimSupplier                    Gold: 500        Warehouse: 500        PASS
DimEmployee                    Gold: 500        Warehouse: 500        PASS
DimPromotion                   Gold: 100        Warehouse: 100        PASS
DimDate                        Gold: 2191       Warehouse: 2191       PASS
FactSales                      Gold: 1000000    Warehouse: 1000000    PASS
FactInventory                  Gold: 250000     Warehouse: 250000     PASS
FactReturns                    Gold: 77065      Warehouse: 77065      PASS
GoldSalesDaily                 Gold: 2191       Warehouse: 2191       PASS
GoldSalesMonthly               Gold: 72         Warehouse: 72        